# Five-minute Urdu audiovisual pilot

Source: official TEDx Talks video VD-RK-nY66k. Exploratory interval: **01:30–06:30**.

This notebook includes acquisition so it can run in a fresh Colab CPU runtime. Run cells in order. The source stays unchanged; processing copies preserve elapsed time. The medium multilingual Whisper model runs locally in Colab with int8 inference. It produces an unreviewed Urdu transcript and estimated word timings, not corrected-text forced alignment or phoneme labels.

The sample includes wide stage shots and close views. No visual interval has been accepted for the dataset, and lip synchronization has not been validated. Rights for a reusable dataset release remain unresolved.

The final stage writes a review video, clickable transcript, correction TSV, raw ASR JSON, word TSV, subtitles, candidate shot boundaries, and a ZIP under `/content/urdu-av-pilot/`. Runtime files are temporary.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp[default]", "imageio-ffmpeg"], check=True)


In [ ]:
from pathlib import Path
import json, shutil, hashlib
from datetime import datetime, timezone
from importlib.metadata import version
import imageio_ffmpeg
from yt_dlp import YoutubeDL

URL = "https://www.youtube.com/watch?v=VD-RK-nY66k"
ROOT = Path("/content/urdu-av-pilot")
RAW = ROOT / "raw"
RAW.mkdir(parents=True, exist_ok=True)
BIN = ROOT / "tools"
BIN.mkdir(exist_ok=True)
FFMPEG = BIN / "ffmpeg"
if not FFMPEG.exists():
    FFMPEG.symlink_to(imageio_ffmpeg.get_ffmpeg_exe())
COMMON = {"noplaylist": True, "socket_timeout": 20, "retries": 1, "extractor_retries": 1}
if shutil.which("node"):
    COMMON["js_runtimes"] = {"node": {}}
print({"yt_dlp": version("yt-dlp"), "imageio_ffmpeg": version("imageio-ffmpeg"), "folder": str(ROOT)})


In [ ]:
with YoutubeDL({**COMMON, "quiet": True}) as ydl:
    info = ydl.extract_info(URL, download=False)
manifest = {
    "source_id": info["id"], "source_url": URL, "title": info.get("title"),
    "channel": info.get("channel"), "duration_seconds": info.get("duration"),
    "platform_language": info.get("language"), "verified_spoken_language": None,
    "dataset_reuse_permission": "not established",
    "review_status": "pending audio, visual, and sync review",
    "pilot_interval": None, "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "tool_versions": {"yt-dlp": version("yt-dlp"), "imageio-ffmpeg": version("imageio-ffmpeg")},
}
print(json.dumps(manifest, ensure_ascii=False, indent=2))
# Save selected metadata only, not expiring stream URLs or account information.
(RAW / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


## Download the observed 1080p video and audio streams

Formats `137` (1080p H.264 video) and `140` (AAC audio without the DRC suffix) were listed for this source during inspection. They may change. If unavailable, inspect the current format list before selecting a replacement; do not silently use an audio-only or low-resolution file. This cell remuxes the streams into one MP4 without changing their playback speed.


In [ ]:
opts = {
    **COMMON, "format": "137+140", "merge_output_format": "mp4",
    "ffmpeg_location": str(FFMPEG),
    "outtmpl": str(RAW / "%(id)s.%(ext)s"),
}
with YoutubeDL(opts) as ydl:
    ydl.download([URL])
VIDEO = RAW / f"{manifest['source_id']}.mp4"
if not VIDEO.is_file():
    raise RuntimeError("Merged MP4 was not created. Inspect the download/merge error above.")
# Decode a short interval to check that the file has readable audio and video.
subprocess.run([str(FFMPEG), "-v", "error", "-i", str(VIDEO), "-t", "2",
                "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"], check=True)
hash_value = hashlib.sha256()
with VIDEO.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        hash_value.update(chunk)
manifest.update({"local_filename": VIDEO.name, "size_bytes": VIDEO.stat().st_size,
                 "sha256": hash_value.hexdigest(), "download_status": "complete"})
(RAW / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {VIDEO} ({VIDEO.stat().st_size / 1024**2:.1f} MiB)")


## Exploratory sample, not accepted corpus material

The assistant selected source seconds 90–390 for the first processing test. Source audio and video start at zero. Review derivatives use `source_seconds = clip_seconds + 90`; no time stretching is applied. Periodic frame inspection found mixed wide and close views. Inspect continuous footage before accepting an interval.


In [ ]:
# Optional: uncomment to save the source and its manifest to your computer.
# from google.colab import files
# files.download(str(VIDEO))
# files.download(str(RAW / "source_manifest.json"))


# Read back the saved acquisition record.
saved = json.loads((RAW / "source_manifest.json").read_text())
assert saved["download_status"] == "complete"
assert (RAW / saved["local_filename"]).stat().st_size == saved["size_bytes"]
print(json.dumps(saved, ensure_ascii=False, indent=2))


In [ ]:
# Five-minute exploratory sample. Original source remains unchanged.
import subprocess, sys, json, shutil, hashlib, time
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1", "ctranslate2==4.8.2", "av==18.1.0"], check=True)
import imageio_ffmpeg
ROOT = Path("/content/urdu-av-pilot")
VIDEO = ROOT / "raw/VD-RK-nY66k.mp4"
assert VIDEO.exists(), "Run acquisition first; runtime files may have expired."
PILOT = ROOT / "pilot_0090_0390"
PILOT.mkdir(exist_ok=True)
START, DURATION = 90.0, 300.0
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
probe = json.loads(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=start_time,duration:stream=index,codec_type,codec_name,start_time,duration,width,height,avg_frame_rate,sample_rate", "-of", "json", str(VIDEO)], text=True))
(PILOT / "source_probe.json").write_text(json.dumps(probe, indent=2))
print(json.dumps(probe, indent=2))
# Resampling changes sample rate, not elapsed time. No denoising or time stretching.
AUDIO = PILOT / "audio_16k.wav"
subprocess.run([FFMPEG, "-y", "-v", "error", "-i", str(VIDEO), "-ss", str(START), "-t", str(DURATION), "-map", "0:a:0", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(AUDIO)], check=True)
import wave
with wave.open(str(AUDIO)) as wf:
    assert wf.getnframes() == 4800000 and wf.getframerate() == 16000
print("Ready: source 01:30–06:30; exactly 300 s mono PCM at 16 kHz")
print({name: version(name) for name in ["faster-whisper", "ctranslate2", "av"]})


In [ ]:
# Draft ASR word timings, NOT corrected-text forced alignment or phoneme labels.
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
import csv
MODEL_ID = "Systran/faster-whisper-medium"
model_path = snapshot_download(MODEL_ID, revision="08e178d48790749d25932bbc082711ddcfdfbc4f")
model_revision = Path(model_path).name
model = WhisperModel(model_path, device="cpu", compute_type="int8", cpu_threads=2)
# Retain the model's language estimate separately from the Urdu decoding setting.
from faster_whisper.audio import decode_audio
samples = decode_audio(str(AUDIO), sampling_rate=16000)
detected_language, language_probability, language_scores = model.detect_language(samples)
print("Automatic language estimate:", detected_language, language_probability, flush=True)
print("Top language scores:", language_scores[:5], flush=True)
started = time.monotonic()
segments, info = model.transcribe(str(AUDIO), language="ur", task="transcribe", beam_size=5,
    word_timestamps=True, vad_filter=True, condition_on_previous_text=False)
records = []
with (PILOT / "segments.partial.jsonl").open("w", encoding="utf-8") as stream:
    for segment in segments:
        record = segment._asdict()
        record["words"] = [w._asdict() for w in segment.words or []]
        records.append(record)
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        print(f"[{segment.start + START:.2f}–{segment.end + START:.2f}] {segment.text}", flush=True)
result = {
    "source_id": "VD-RK-nY66k", "source_start_seconds": START, "duration_seconds": DURATION,
    "time_mapping": "source_seconds = clip_seconds + 90.0; no temporal warp",
    "model_id": MODEL_ID, "model_revision": model_revision, "device": "cpu", "compute_type": "int8",
    "tool_versions": {n: version(n) for n in ["faster-whisper", "ctranslate2", "av"]},
    "decoding": {"language": "ur", "task": "transcribe", "beam_size": 5, "word_timestamps": True,
                 "vad_filter": True, "condition_on_previous_text": False},
    "automatic_language": {"language": detected_language, "probability": language_probability,
                           "top_scores": language_scores[:5]},
    "review_status": "unreviewed automatic transcript and estimated word boundaries",
    "phoneme_alignment": None, "sync_review": "not performed", "human_language_review": "pending",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.monotonic() - started, 2), "segments": records,
}
(PILOT / "transcript_raw.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
fields = ["occurrence_id", "segment_id", "word_raw", "clip_start", "clip_end", "source_start", "source_end", "probability", "review_status"]
words = []
for s in records:
    for w in s["words"]:
        words.append(dict(zip(fields, [f"VD-RK-nY66k_0090_w{len(words):05d}", s["id"], w["word"],
            w["start"], w["end"], round(START + w["start"], 3), round(START + w["end"], 3),
            w["probability"], "unreviewed_asr_estimate"])))
with (PILOT / "words_draft.tsv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fields, delimiter="\t"); writer.writeheader(); writer.writerows(words)
(PILOT / "transcript_draft.txt").write_text("\n".join(s["text"].strip() for s in records), encoding="utf-8")
print("COMPLETE", {"segments": len(records), "word_occurrences": len(words), "elapsed_seconds": result["elapsed_seconds"], "folder": str(PILOT)}, flush=True)


In [ ]:
# Structural checks and a review bundle. Passing these checks is not annotation validation.
import math, re, html, zipfile
assert records and words, "ASR produced no usable draft."
issues = []
previous_end = 0.0
for w in words:
    a, b = w["clip_start"], w["clip_end"]
    reasons = []
    if not all(math.isfinite(x) for x in [a, b]) or not 0 <= a <= b <= DURATION + 0.05:
        reasons.append("invalid_bounds")
    if b - a < 0.02: reasons.append("very_short_or_zero_duration")
    if a < previous_end - 0.02: reasons.append("overlapping_previous_word")
    if w["probability"] < 0.5: reasons.append("low_model_probability")
    if reasons: issues.append({"occurrence_id": w["occurrence_id"], "word": w["word_raw"], "reasons": reasons})
    previous_end = b
assert not any("invalid_bounds" in x["reasons"] for x in issues), "Invalid timestamps require repair."
def stamp(t, sep=","):
    ms = round(t * 1000); h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"
srt = "\n\n".join(f"{i+1}\n{stamp(s['start'])} --> {stamp(s['end'])}\n{s['text'].strip()}" for i, s in enumerate(records))
(PILOT / "transcript_draft.srt").write_text(srt + "\n", encoding="utf-8")
with (PILOT / "utterances_for_review.tsv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["segment_id", "source_start", "source_end", "raw_text", "corrected_text", "reviewer", "notes"])
    for s in records: writer.writerow([s["id"], START+s["start"], START+s["end"], s["text"].strip(), "", "", ""])
# Candidate visual cuts; this heuristic can miss cuts and is not an acceptance mask.
scene = subprocess.run([FFMPEG, "-hide_banner", "-ss", str(START), "-t", str(DURATION), "-i", str(VIDEO),
    "-vf", "scale=320:180,select='gt(scene,0.3)',showinfo", "-an", "-f", "null", "-"], capture_output=True, text=True, check=True)
cut_times = [round(float(t) + START, 3) for t in re.findall(r"pts_time:([0-9.]+)", scene.stderr)]
(PILOT / "scene_candidates.json").write_text(json.dumps({"method": "FFmpeg scene > 0.3 at 320x180", "source_seconds": cut_times, "review_status": "unreviewed candidate cuts"}, indent=2))
summary = {"sample_source_start": START, "sample_source_end": START+DURATION, "segments": len(records),
    "word_occurrences": len(words), "flagged_words": len(issues), "flags": issues, "candidate_scene_cuts": len(cut_times),
    "structural_bounds_check": "passed", "transcript_accuracy": "not measured; human correction pending",
    "word_boundary_accuracy": "not measured; ASR estimates only", "phoneme_labels": "not produced",
    "audiovisual_sync": "not validated", "visual_eligibility": "mixed close and wide shots; no interval accepted yet"}
(PILOT / "quality_report.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
# Review derivative; retain original media as the authoritative recording.
PREVIEW = PILOT / "review.mp4"
subprocess.run([FFMPEG, "-y", "-v", "error", "-i", str(VIDEO), "-ss", str(START), "-t", str(DURATION),
    "-map", "0:v:0", "-map", "0:a:0", "-vf", "scale=1280:720", "-c:v", "libx264", "-preset", "veryfast", "-crf", "24",
    "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", str(PREVIEW)], check=True)
subprocess.run([FFMPEG, "-y", "-v", "error", "-ss", str(START), "-t", str(DURATION), "-i", str(VIDEO),
    "-vf", "select='not(mod(n,300))',scale=320:180,tile=5x6", "-frames:v", "1", str(PILOT / "contact_sheet.jpg")], check=True)
rows = "".join(f'<tr><td><button onclick="seek({s["start"]})">{START+s["start"]:.2f}–{START+s["end"]:.2f}</button></td><td dir="rtl">{html.escape(s["text"].strip())}</td></tr>' for s in records)
page = '<!doctype html><meta charset="utf-8"><title>Urdu pilot draft review</title><style>body{font:18px system-ui;max-width:1000px;margin:30px auto}video{width:100%}td{padding:12px;border-bottom:1px solid #ddd}button{cursor:pointer}</style><h1>Urdu pilot: unreviewed draft</h1><p>Source 01:30–06:30. Buttons show original source seconds. Word timings are ASR estimates, not validated alignments. Wide shots need filtering.</p><video id="v" controls src="review.mp4"></video><p>Correct the separate utterances_for_review.tsv file. Preserve the raw transcript.</p><table>'+rows+'</table><script>function seek(t){const v=document.getElementById("v");v.currentTime=t;v.play();}</script>'
(PILOT / "review.html").write_text(page, encoding="utf-8")
(PILOT / "README.txt").write_text("Exploratory Urdu sample: source seconds 90 through 390. Open review.html beside review.mp4. All transcripts/timings are automatic and unreviewed. Correct utterances_for_review.tsv; never overwrite transcript_raw.json. Candidate scene cuts are not verified. No phoneme annotations, accepted visual intervals, or synchronization validation yet. Rights for redistribution remain unresolved.\n", encoding="utf-8")
BUNDLE = ROOT / "urdu_pilot_0090_0390_review.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(PILOT.iterdir()):
        if f.is_file() and f.name != "segments.partial.jsonl": z.write(f, f.name)
print(json.dumps({k:v for k,v in summary.items() if k != "flags"}, ensure_ascii=False, indent=2))
print("MODEL REVISION", model_revision)
print("BUNDLE READY", str(BUNDLE), BUNDLE.stat().st_size)


In [ ]:
# Download the completed review bundle before this runtime expires.
from google.colab import files
assert BUNDLE.is_file()
files.download(str(BUNDLE))
